# Chapitre 6 — Structuration et transformation

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Restructurer** des DataFrames avec pivot et melt selon les besoins d'analyse
2. **Combiner** des sources de données avec merge et concat en choisissant la bonne méthode
3. **Agréger** des données avec groupby et créer des statistiques par groupe
4. **Créer** de nouvelles variables pertinentes (feature engineering) pour l'analyse et le ML

---

## 6.3 Agrégations avec GroupBy

In [1]:
import pandas as pd
import numpy as np

# Données de ventes
np.random.seed(42)
df = pd.DataFrame({
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 100),
    'categorie': np.random.choice(['A', 'B', 'C'], 100),
    'ventes': np.random.randint(50, 500, 100),
    'client_id': np.random.randint(1, 30, 100)
})

print("Données :")
print(df.head(10))

Données :
  region categorie  ventes  client_id
0    Est         C     333          3
1  Ouest         B      77         19
2   Nord         B     157         20
3    Est         B      93          7
4    Est         B     389         20
5  Ouest         B     335          9
6   Nord         B     495          1
7   Nord         C     380          8
8    Est         C     177          7
9    Sud         B     397         18


In [ ]:
# Agrégation simple
print("Ventes totales par région :")
print(df.groupby('region')['ventes'].sum())

# regrouper les données du DataFrame df selon la colonne region, 
# puis calcule la somme des ventes pour chaque région. 
# Le résultat est une série où chaque région est associée au total de ses ventes.

Ventes totales par région :
region
Est      6792
Nord     5368
Ouest    8994
Sud      7936
Name: ventes, dtype: int64


In [5]:
# Plusieurs colonnes de groupement
print("\nVentes par région et catégorie :")
print(df.groupby(['region', 'categorie'])['ventes'].sum())
      #.unstack())
# unstack transforme les lignes en colonnes


Ventes par région et catégorie :
region  categorie
Est     A            2324
        B            1760
        C            2708
Nord    A            1516
        B            2320
        C            1532
Ouest   A            4038
        B            1280
        C            3676
Sud     A            3094
        B            2533
        C            2309
Name: ventes, dtype: int64


In [4]:
# Plusieurs fonctions d'agrégation
print("\nStatistiques par région :")
print(df.groupby('region')['ventes'].agg(['sum', 'mean', 'count', 'std']).round(2))


Statistiques par région :
         sum    mean  count     std
region                             
Est     6792  283.00     24  124.75
Nord    5368  268.40     20  118.19
Ouest   8994  299.80     30  121.96
Sud     7936  305.23     26  104.31


In [6]:
# Agrégations nommées (syntaxe moderne)
df_agg = df.groupby('region').agg(
    total_ventes=('ventes', 'sum'),
    moyenne_ventes=('ventes', 'mean'),
    nb_transactions=('ventes', 'count'),
    nb_clients=('client_id', 'nunique')
).round(2).reset_index()

print("\nAgrégations nommées :")
print(df_agg)


Agrégations nommées :
  region  total_ventes  moyenne_ventes  nb_transactions  nb_clients
0    Est          6792          283.00               24          15
1   Nord          5368          268.40               20          11
2  Ouest          8994          299.80               30          21
3    Sud          7936          305.23               26          19


### Fonctions d'agrégation courantes

| Fonction | Description |
|----------|-------------|
| `sum()` | Somme |
| `mean()` | Moyenne |
| `median()` | Médiane |
| `count()` | Nombre de valeurs non-null |
| `size()` | Nombre total de lignes |
| `nunique()` | Nombre de valeurs uniques |
| `min()`, `max()` | Minimum, Maximum |
| `std()`, `var()` | Écart-type, Variance |
| `first()`, `last()` | Première, Dernière valeur |

### ✍️ Exercice 6.5 : Analyse par groupe (15 min)

In [7]:
import pandas as pd
import numpy as np

# Données de ventes
np.random.seed(42)
df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=1000, freq='D'),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 1000),
    'categorie': np.random.choice(['Électronique', 'Vêtements', 'Alimentation'], 1000),
    'montant': np.random.randint(10, 500, 1000),
    'client_id': np.random.randint(1, 200, 1000)
})
df['mois'] = df['date'].dt.month

print(f"Dataset : {len(df)} lignes")

Dataset : 1000 lignes


In [8]:
# 1. Ventes totales par région
print("Ventes par région :")
print(df.groupby('region')['montant'].sum())

Ventes par région :
region
Est      61044
Nord     67003
Ouest    71729
Sud      56810
Name: montant, dtype: int64


In [9]:
# 2. Ventes moyennes par catégorie et région
print("\nVentes moyennes par catégorie et région :")
print(df.groupby(['categorie', 'region'])['montant'].mean().round(2).unstack())


Ventes moyennes par catégorie et région :
region           Est    Nord   Ouest     Sud
categorie                                   
Alimentation  235.64  260.04  274.91  221.05
Vêtements     272.21  260.57  250.29  274.44
Électronique  278.03  258.34  243.09  251.31


In [10]:
# 3. Nombre de clients uniques par région
print("\nClients uniques par région :")
print(df.groupby('region')['client_id'].nunique())


Clients uniques par région :
region
Est      141
Nord     140
Ouest    157
Sud      136
Name: client_id, dtype: int64


In [11]:
# 4. Top 3 des mois par ventes totales
print("\nTop 3 mois :")
ventes_mois = df.groupby('mois')['montant'].sum().sort_values(ascending=False)
print(ventes_mois.head(3))


Top 3 mois :
mois
5    24354
4    23978
8    23939
Name: montant, dtype: int64


In [12]:
# 5. Rapport complet par région
rapport = df.groupby('region').agg(
    total_ventes=('montant', 'sum'),
    moyenne_ventes=('montant', 'mean'),
    nb_transactions=('montant', 'count'),
    nb_clients=('client_id', 'nunique')
).round(2)

print("\nRapport complet :")
print(rapport)


Rapport complet :
        total_ventes  moyenne_ventes  nb_transactions  nb_clients
region                                                           
Est            61044          263.12              232         141
Nord           67003          259.70              258         140
Ouest          71729          256.18              280         157
Sud            56810          247.00              230         136
